# ColabBridge — ResNet-50 Image Classifier

Classifies images using ResNet-50 on the Colab T4 GPU.
A drag-and-drop web UI connects to this server via ColabBridge.

**Steps:**
1. Runtime → Change Runtime Type → **GPU: T4**
2. Replace `YOUR_REGISTRY_URL` in Cell 2
3. Run Cell 1 (install), then Cell 2 (start server)
4. Open `examples/image_classifier/web/index.html` in your browser

In [ ]:
# Cell 1 — Install dependencies
!pip install -q colabbridge-server torch torchvision

In [ ]:
# Cell 2 — Start classifier server
import base64, io, torch
from PIL import Image
from torchvision.models import resnet50, ResNet50_Weights
from colabbridge import ColabBridge

bridge = ColabBridge(
    registry_url="YOUR_REGISTRY_URL",  # e.g. https://your-registry.up.railway.app
)

model = None
weights = None

def load_models():
    global model, weights
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading ResNet-50 on {device}...")
    weights = ResNet50_Weights.IMAGENET1K_V2
    model = resnet50(weights=weights).to(device).eval()
    with torch.no_grad():
        model(torch.randn(1, 3, 224, 224).to(device))
    print("Ready!")

@bridge.post("/classify")
def classify(image_b64: str, top_k: int = 5) -> dict:
    device = next(model.parameters()).device
    image = Image.open(io.BytesIO(base64.b64decode(image_b64))).convert("RGB")
    tensor = weights.transforms()(image).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    top = probs.topk(top_k)
    cats = weights.meta["categories"]
    return {
        "predictions": [
            {"label": cats[i], "score": round(s.item(), 4)}
            for i, s in zip(top.indices.tolist(), top.values.tolist())
        ]
    }

bridge.run(warmup=load_models)